# 05 - Afinidade de categorias e desempenho de vendedores

Aqui eu fecho a analise olhando para categorias compradas juntas, desempenho de vendedores e um resumo geral do projeto.

In [ ]:
from pathlib import Path
import sqlite3
import urllib.request
import zipfile

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

PROJECT_DIR = Path.cwd()
DB_ZIP_URL = "https://github.com/Urpia-S/Portfolio_/releases/download/data-v1/olist_colab.sqlite.zip"
OUTPUT_DIR = PROJECT_DIR / "outputs_colab"
DB_PATH = PROJECT_DIR / "olist_colab.sqlite"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def baixar_banco_da_release():
    zip_path = PROJECT_DIR / "olist_colab.sqlite.zip"
    urllib.request.urlretrieve(DB_ZIP_URL, zip_path)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(PROJECT_DIR)

    print("Banco extraido em:", DB_PATH)


if not DB_PATH.exists():
    baixar_banco_da_release()

conn = sqlite3.connect(DB_PATH)


def consulta(sql):
    return pd.read_sql_query(sql, conn)


def salvar_consulta(sql, arquivo):
    df = consulta(sql)
    destino = OUTPUT_DIR / arquivo
    df.to_csv(destino, index=False)
    print(f"Arquivo salvo: {destino}")
    return df


def grafico_barras(df, x, y, titulo, rotacao=0, top=None):
    dados = df.head(top) if top else df
    ax = dados.plot(kind="bar", x=x, y=y, legend=False, figsize=(10, 4))
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.xticks(rotation=rotacao, ha="right" if rotacao else "center")
    plt.tight_layout()
    plt.show()


consulta("""
-- Objetos disponiveis no banco preparado.
SELECT
    type AS tipo,
    name AS objeto
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
LIMIT 20;
""")

## 1. Afinidade e vendedores

Primeiro identifico pares de categorias no mesmo pedido. Depois comparo vendedores por receita, atraso e reviews.

In [ ]:
afinidade_categorias = salvar_consulta("""
-- Crio uma lista unica de categorias por pedido.
WITH categorias_por_pedido AS (
    SELECT DISTINCT
        order_id,
        categoria_ingles
    FROM vw_order_items_enriched
    WHERE categoria_ingles IS NOT NULL
),
-- Considero apenas pedidos com mais de uma categoria.
pedidos_multicategoria AS (
    SELECT order_id
    FROM categorias_por_pedido
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
-- Monto pares sem duplicar combinacoes invertidas.
SELECT
    a.categoria_ingles AS categoria_a,
    b.categoria_ingles AS categoria_b,
    COUNT(*) AS pedidos_em_comum
FROM categorias_por_pedido a
JOIN categorias_por_pedido b
    ON b.order_id = a.order_id
   AND b.categoria_ingles > a.categoria_ingles
JOIN pedidos_multicategoria m
    ON m.order_id = a.order_id
GROUP BY a.categoria_ingles, b.categoria_ingles
HAVING COUNT(*) >= 3
ORDER BY pedidos_em_comum DESC, categoria_a, categoria_b;
""", "afinidade_categorias.csv")

afinidade_categorias.head(10)

In [ ]:
desempenho_vendedores = salvar_consulta("""
-- Agrego receita por vendedor e pedido antes de juntar entrega/review.
WITH vendedor_pedido AS (
    SELECT
        i.seller_id,
        i.order_id,
        SUM(i.price) AS receita_produtos,
        SUM(i.freight_value) AS receita_frete
    FROM vw_order_items_enriched i
    GROUP BY i.seller_id, i.order_id
)
-- Ranking de vendedores com receita, atraso, nota media e classificacao operacional.
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT vp.order_id) AS pedidos,
    ROUND(SUM(vp.receita_produtos), 2) AS receita_produtos,
    ROUND(SUM(vp.receita_frete), 2) AS receita_frete,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN d.entrega_atrasada = 1 THEN vp.order_id END)
        / NULLIF(COUNT(DISTINCT CASE WHEN d.order_delivered_customer_date IS NOT NULL THEN vp.order_id END), 0),
        2
    ) AS percentual_atraso,
    ROUND(AVG(d.dias_atraso), 2) AS media_dias_atraso,
    ROUND(AVG(r.nota_media_review), 2) AS nota_media_review,
    ROUND(100.0 * SUM(r.reviews_baixos) / NULLIF(SUM(r.quantidade_reviews), 0), 2) AS percentual_reviews_baixos,
    CASE
        WHEN COUNT(DISTINCT vp.order_id) >= 50
         AND (
            100.0 * COUNT(DISTINCT CASE WHEN d.entrega_atrasada = 1 THEN vp.order_id END)
            / NULLIF(COUNT(DISTINCT CASE WHEN d.order_delivered_customer_date IS NOT NULL THEN vp.order_id END), 0)
         ) >= 15
         AND AVG(r.nota_media_review) < 4
        THEN 'alto_volume_com_risco'
        ELSE 'monitorar_padrao'
    END AS classificacao_operacional
FROM vendedor_pedido vp
JOIN core_sellers s ON s.seller_id = vp.seller_id
JOIN vw_order_delivery d ON d.order_id = vp.order_id
LEFT JOIN vw_reviews_por_pedido r ON r.order_id = vp.order_id
GROUP BY s.seller_id, s.seller_city, s.seller_state
ORDER BY receita_produtos DESC;
""", "desempenho_vendedores.csv")

desempenho_vendedores.head()

## 2. Resumo executivo

Por fim, consolido alguns indicadores centrais em uma unica tabela.

In [ ]:
resumo = consulta("""
-- Resumo geral do projeto em uma unica linha.
WITH item_totals AS (
    SELECT SUM(price) AS receita_produtos
    FROM core_order_items
),
order_counts AS (
    SELECT
        COUNT(*) AS pedidos_totais,
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS pedidos_entregues
    FROM core_orders
),
delivery AS (
    SELECT
        AVG(dias_entrega) AS media_dias_entrega,
        100.0 * SUM(CASE WHEN entrega_atrasada = 1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END), 0) AS percentual_atraso
    FROM vw_order_delivery
),
reviews AS (
    SELECT AVG(review_score) AS nota_media_geral
    FROM core_order_reviews
)
SELECT
    pedidos_totais,
    pedidos_entregues,
    ROUND(receita_produtos, 2) AS receita_produtos,
    ROUND(receita_produtos / pedidos_totais, 2) AS ticket_medio_produtos_por_pedido,
    ROUND(media_dias_entrega, 2) AS media_dias_entrega,
    ROUND(percentual_atraso, 2) AS percentual_atraso,
    ROUND(nota_media_geral, 2) AS nota_media_geral
FROM order_counts, item_totals, delivery, reviews;
""")

resumo